# Tagging and Extraction Using OpenAI functions

In [ ]:
# 读取 .env 里的 OPENAI_API_KEY
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

In [ ]:
# pydantic 用来定义带类型校验的数据模型
# 【版本兼容修复】langchain.utils.openai_functions 已不存在，
# convert_pydantic_to_openai_function 现在在 langchain_community.utils.openai_functions 下
from typing import List
from pydantic import BaseModel, Field
from langchain_community.utils.openai_functions import convert_pydantic_to_openai_function

In [ ]:
# "Tagging"（打标签）：不是提取新信息，而是给已有文本打上分类标签（情感、语言等）
# 这是 function calling 用来做"结构化分类"的经典用法
class Tagging(BaseModel):
        """Tag the piece of text with particular info."""
        sentiment: str = Field(description="sentiment of text, should be `pos`, `neg`, or `neutral`")
        language: str = Field(description="language of text (should be ISO 639-1 code)")

In [ ]:
# 看看转换出来的 OpenAI function schema 长什么样
convert_pydantic_to_openai_function(Tagging)

In [ ]:
# 【版本兼容修复】langchain.prompts / langchain.chat_models 已不存在，
# 分别搬到了 langchain_core.prompts 和 langchain_openai
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [ ]:
# temperature=0 让输出更确定、更可复现，适合分类/抽取这种需要稳定结构化结果的任务
model = ChatOpenAI(temperature=0)

In [ ]:
tagging_functions=[convert_pydantic_to_openai_function(Tagging)]

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Think carefully, and then tag the text as instructed"),
    ("user", "{input}")
])

In [ ]:
# 强制模型必须调用 Tagging 这个函数，这样每次输出都会是结构化的 sentiment/language，而不是自由文本
model_with_functions = model.bind(
    functions=tagging_functions,
    function_call={"name": "Tagging"}
)

In [ ]:
tagging_chain = prompt | model_with_functions

In [ ]:
# 期望：sentiment=pos, language=en
tagging_chain.invoke({"input": "I love langchain"})

In [ ]:
# 期望：sentiment=neg（"我不喜欢这个食物"）, language=it（意大利语）
tagging_chain.invoke({"input": "non mi piace questo cibo"})

In [ ]:
# JsonOutputFunctionsParser：把模型返回的 function_call.arguments（JSON 字符串）自动解析成 Python dict，
# 免去我们手动 json.loads 的步骤
# 【版本兼容修复】langchain.output_parsers 已不存在，这个 parser 现在在 langchain_core.output_parsers.openai_functions 下
from langchain_core.output_parsers.openai_functions import JsonOutputFunctionsParser

In [ ]:
# 把 JsonOutputFunctionsParser 接到 chain 末尾，这样 invoke() 直接返回 dict，而不是原始的消息对象
tagging_chain = prompt | model_with_functions | JsonOutputFunctionsParser()

In [ ]:
# 现在直接返回 {"sentiment": "neg", "language": "it"} 这样的 dict，而不是消息对象
tagging_chain.invoke({"input": "non mi piace questo cibo"})

In [ ]:
from typing import Optional

# age 是可选字段：不是所有文本里都会提到年龄，所以要允许它缺失
class Person(BaseModel):
    """Information about a person."""
    name: str = Field(description="person's name")
    # 【真实 bug 修复 / pydantic 版本差异】在 pydantic v1（课程录制时）里，
    # Optional[int] 会自动隐含 default=None；但 pydantic v2（当前环境）不再有这个隐式行为，
    # 如果不显式写 default=None，age 仍然会被当成"必填"字段——
    # 转换成 OpenAI function schema 后，"required" 列表里会错误地包含 "age"，
    # 这就违背了"允许缺失年龄"的设计初衷。必须显式加上 default=None。
    age: Optional[int] = Field(default=None, description="person's age")

In [ ]:
# Information 里嵌套了 Person 列表，一次可以抽取"多个人"的信息
class Information(BaseModel):
    """Information to extract."""
    people: List[Person] = Field(description="List of info about people")

In [ ]:
# 修复后，required 里应该只有 "people"（Information 层面），
# 内层 Person 的 required 里也应该只有 "name"，"age" 不再被错误地标为必填
convert_pydantic_to_openai_function(Information)

In [ ]:
# 这是一个"信息抽取"场景：强制模型调用 Information 函数，把非结构化文本转成结构化数据
extraction_functions = [convert_pydantic_to_openai_function(Information)]
extraction_model = model.bind(functions=extraction_functions, function_call={"name": "Information"})

In [ ]:
# 期望：抽取出 Joe（age=30）和 Martha（age 未提及，应为 None 而不是被模型瞎猜一个数字）
extraction_model.invoke("Joe is 30, his mom is Martha")

In [ ]:
# system prompt 明确要求"不要瞎猜"，配合上面 age 字段允许缺失，模型应该老老实实留空未知信息
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the relevant information, if not explicitly provided do not guess. Extract partial info"),
    ("human", "{input}")
])

In [ ]:
extraction_chain = prompt | extraction_model

In [ ]:
extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"})

In [ ]:
# 接上 JsonOutputFunctionsParser，直接拿到 {"people": [...]} 这样的 dict
extraction_chain = prompt | extraction_model | JsonOutputFunctionsParser()

In [ ]:
extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"})

In [ ]:
# JsonKeyOutputFunctionsParser：在 JsonOutputFunctionsParser 基础上，再自动取出指定 key 对应的值
# （比如直接拿到 people 列表，而不是 {"people": [...]} 这层外壳）
# 【版本兼容修复】同样从 langchain_core.output_parsers.openai_functions 导入
from langchain_core.output_parsers.openai_functions import JsonKeyOutputFunctionsParser

In [ ]:
extraction_chain = prompt | extraction_model | JsonKeyOutputFunctionsParser(key_name="people")

In [ ]:
# 现在直接返回 people 列表本身：[{"name": "Joe", "age": 30}, {"name": "Martha", "age": None}]
extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"})

In [ ]:
# 【版本兼容修复】langchain.document_loaders 已不存在，文档加载器搬到了 langchain_community.document_loaders
# 设置 USER_AGENT 环境变量，避免 WebBaseLoader 发请求时报 "USER_AGENT not set" 的警告
os.environ.setdefault("USER_AGENT", "DeepLearningAI-LangChain-Course/1.0")
from langchain_community.document_loaders import WebBaseLoader
# 从真实网页抓取一篇技术博客的正文内容，作为后面信息抽取的语料（这一步走的是普通 HTTP 请求，不需要 OpenAI key）
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
documents = loader.load()

In [ ]:
doc = documents[0]

In [ ]:
# 只取前 10000 字符，避免超出模型的上下文窗口 / 减少 token 消耗
page_content = doc.page_content[:10000]

In [ ]:
print(page_content[:1000])

In [ ]:
# 用来对整篇文章做"打标签"式总结：摘要、语言、关键词
class Overview(BaseModel):
    """Overview of a section of text."""
    summary: str = Field(description="Provide a concise summary of the content.")
    language: str = Field(description="Provide the language that the content is written in.")
    keywords: str = Field(description="Provide keywords related to the content.")

In [ ]:
overview_tagging_function = [
    convert_pydantic_to_openai_function(Overview)
]
tagging_model = model.bind(
    functions=overview_tagging_function,
    function_call={"name":"Overview"}
)
tagging_chain = prompt | tagging_model | JsonOutputFunctionsParser()

In [ ]:
tagging_chain.invoke({"input": page_content})

In [ ]:
class Paper(BaseModel):
    """Information about papers mentioned."""
    title: str
    # 【真实 bug 修复 / pydantic 版本差异】同上面 Person.age 的问题：
    # pydantic v2 里 Optional[str] 不再隐式等于"默认值是 None"，必须显式写 = None，
    # 否则 author 会被错误地标记为必填字段
    author: Optional[str] = None


class Info(BaseModel):
    """Information to extract"""
    papers: List[Paper]

In [ ]:
# 强制模型调用 Info，并且用 JsonKeyOutputFunctionsParser 直接拿到 papers 列表
paper_extraction_function = [
    convert_pydantic_to_openai_function(Info)
]
extraction_model = model.bind(
    functions=paper_extraction_function,
    function_call={"name":"Info"}
)
extraction_chain = prompt | extraction_model | JsonKeyOutputFunctionsParser(key_name="papers")

In [ ]:
# 注意：这里用的 prompt 还是前面"抽取人物信息"的 prompt（没有专门针对论文抽取写 system 消息），
# 所以效果可能一般——下面几个 cell 会专门写一个更贴切的 prompt 来改进
extraction_chain.invoke({"input": page_content})

In [ ]:
# 专门为"论文抽取"任务写的 system prompt：强调不要编造信息、没提到就返回空列表
template = """A article will be passed to you. Extract from it all papers that are mentioned by this article follow by its author.

Do not extract the name of the article itself. If no papers are mentioned that's fine - you don't need to extract any! Just return an empty list.

Do not make up or guess ANY extra information. Only extract what exactly is in the text."""

prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", "{input}")
])

In [ ]:
# 用新的 prompt 重新组装 chain
extraction_chain = prompt | extraction_model | JsonKeyOutputFunctionsParser(key_name="papers")

In [ ]:
extraction_chain.invoke({"input": page_content})

In [ ]:
# 输入跟论文完全无关，期望返回空列表 []，而不是瞎编一篇论文出来
extraction_chain.invoke({"input": "hi"})

In [ ]:
# 【版本兼容修复】langchain.text_splitter 已不存在，文本切分器现在在独立的 langchain_text_splitters 包里
# RecursiveCharacterTextSplitter 把长文本切成多个小块（chunk），避免超出模型上下文长度限制
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_overlap=0)

In [ ]:
# 这次对完整的 doc.page_content（没有截断到 10000 字符）做切分，得到多个文本块
splits = text_splitter.split_text(doc.page_content)

In [ ]:
len(splits)

In [ ]:
# flatten：把"列表的列表"拍平成一维列表，因为对多个文本块分别抽取后会得到 [[...], [...], ...] 这样的结构
def flatten(matrix):
    flat_list = []
    for row in matrix:
        flat_list += row
    return flat_list

In [ ]:
flatten([[1, 2], [3, 4]])

In [ ]:
print(splits[0])

In [ ]:
# 【版本兼容修复】langchain.schema.runnable 已不存在，RunnableLambda 现在在 langchain_core.runnables 下
# RunnableLambda 把一个普通 Python 函数包装成 Runnable，这样它就能参与 LCEL 的 | 组合
from langchain_core.runnables import RunnableLambda

In [ ]:
# prep：把一整篇长文本切分成多个 chunk，并且包装成 extraction_chain 期望的输入格式 {"input": chunk}
prep = RunnableLambda(
    lambda x: [{"input": doc} for doc in text_splitter.split_text(x)]
)

In [ ]:
prep.invoke("hi")

In [ ]:
# 完整流水线：切分成多个 chunk -> 对每个 chunk 分别跑 extraction_chain（.map() 会对列表里每个元素分别调用）
# -> 把多个 chunk 各自抽取出的 papers 列表拍平成一个列表
chain = prep | extraction_chain.map() | flatten

In [ ]:
# 对整篇文章（所有 chunk）跑一遍完整抽取流程
chain.invoke(doc.page_content)